# Predict drifter trajectories with the shipped CoDiT model

This notebook runs the shipped `codit_GRU_open_ocean` checkpoint (72 h horizon)
on data you download for a small region. It has three parts:

1. **Canned demo** — predict on a small pre-configured open-ocean region and
   compare against a real drifter trajectory.
2. **Your own region** — same flow, with a wrapper function you call with
   your own bounding box and dates.
3. **Point-and-shoot** — pick a single (latitude, longitude, datetime) and
   forecast from there. No ground-truth drifter required.

Prerequisites:
- `pip install -e .` (or `conda env create -f environment.yml`).
- `config.ini` populated with Copernicus and CDS credentials (see the
  project README).
- `CODIT_DATA_PATH` set to a writeable directory, or unset (defaults to
  `<repo>/data`).

Downloads take a few minutes per region because HYCOM, ERA5, Copernicus,
and the NOAA drifter archive are queried in turn. Files are cached on disk;
re-running skips completed downloads.


In [ ]:
import os
from datetime import datetime, timedelta

import jax
import numpy as np
from jax import numpy as jnp
from matplotlib import pyplot as plt
from cartopy import crs as ccrs
import cartopy.feature as cfeature

from codit.data_module import config_funcs, config_paths, dataset_download
from codit.data_module.dataset import OceanDataset
from codit.model_module.diffusion_model import CoDiTModel

np.random.seed(0)
key = jax.random.PRNGKey(0)

## 1. Predict on a canned demo region

The shipped `codit_GRU_open_ocean` model was trained on subtropical gyres
with a 72-hour horizon. We use a small (~3° square) demo box in the North
Atlantic, over one week in mid-2021. That window is short
enough for the download to complete quickly.

### Load the model

The checkpoint ships with the repo under `data/trained_models/`. It stores
its own hyperparameters (K = 100 diffusion steps, GRU denoiser, three
context variables: HYCOM currents, Copernicus/GLORYS currents, ERA5 wind).


In [ ]:
model = CoDiTModel.import_trained_model("codit_GRU_open_ocean")
print("Trained on vars:", model.params["vars"])
print("Prediction length (hours):", model.params["prediction_length"])

### Define the demo region and download the four sources

Each `download_*` call writes into `dataset_path`. If files already exist
(from a previous run), the download is skipped.


In [ ]:
DEMO_NAME = "predict_demo_atlantic"
CENTER_LAT, CENTER_LON = 32.0, -45.0     # subtropical N. Atlantic
COORD_DELTA = 1.5                         # half-width in degrees (~3° box)
CONTEXT_DELTA = 0.5                       # padding around the box for context
START = datetime(2021, 6, 1)
END = datetime(2021, 6, 8)                # 1 week — enough to sample one 72h trajectory
PREDICTION_LENGTH = model.params["prediction_length"]

min_lat, max_lat = CENTER_LAT - COORD_DELTA, CENTER_LAT + COORD_DELTA
min_lon, max_lon = CENTER_LON - COORD_DELTA, CENTER_LON + COORD_DELTA

cfg_dir = os.path.join(config_paths.CONFIG_PATH, "example_datasets")
os.makedirs(cfg_dir, exist_ok=True)

raw = {
    "name": DEMO_NAME,
    "latitudes": [min_lat, max_lat],
    "longitudes": [min_lon, max_lon],
    "start_date": START,
    "end_date": END,
    "context_delta": CONTEXT_DELTA,
    "description": "canned demo region for 01_predict_quickstart",
}
config_funcs.save_config(raw, cfg_dir, config_paths.DATA_PATH)
config = config_funcs.load_config(DEMO_NAME, cfg_dir)
print("Data will be written under:", config["dataset_path"])

In [ ]:
dataset_download.download_drifter(config)
dataset_download.download_hycom(config)
dataset_download.download_era5(config)
dataset_download.download_copernicus(config)

### Wrap the raw NetCDFs in an `OceanDataset` and run one prediction

`OceanDataset[i]` returns everything the model needs for the *i*-th valid
starting position it can extract from the drifter archive: the start lat/lon,
the start time, the full 73-timestep time axis, an `interpax.Interpolator3D`
covering the space-time context, and the ground-truth 72 h trajectory.

`model.predict(...)` samples `num_samples` independent trajectories in one
JIT-compiled sweep. The first sample compiles the JAX graph (30–60 s), and
subsequent samples are cheap.


In [ ]:
test_ds = OceanDataset(
    config,
    length=PREDICTION_LENGTH,
    num_grid_points=16,
    delta_metres=10000,
    interp_method="linear",
)
print(f"Valid starting positions in the demo region: {len(test_ds)}")
if len(test_ds) == 0:
    raise RuntimeError(
        "No drifter starts in this region/window. "
        "Widen the date range or the bounding box."
    )

In [ ]:
# Pick the first valid drifter start and predict 20 sample trajectories.
start_position, start_time, times, interpolator, true_coords = test_ds[0]
print(f"Start position: ({start_position[0]:.3f}, {start_position[1]:.3f})")
print(f"Start time:     {start_time}")

pred_coords = model.predict(
    start_position,
    start_time,
    PREDICTION_LENGTH,
    interpolator,
    test_ds,
    num_samples=20,
    key=key,
)
print(f"Predicted trajectory tensor shape: {pred_coords.shape}  # (samples, hours, [lat, lon])")

### Plot ground truth vs. predicted samples

Grey lines are individual samples from the diffusion model, red is the
per-timestep mean, black is the true drifter trajectory.


In [ ]:
def plot_prediction(pred_coords, true_coords=None, start_position=None, title=""):
    """Plot predicted trajectory samples, optionally overlaying a ground truth."""
    fig = plt.figure(figsize=(7, 7))
    ax = plt.axes(projection=ccrs.PlateCarree())
    ax.add_feature(cfeature.OCEAN, facecolor="#d9edf7")
    ax.add_feature(cfeature.LAND, facecolor="#f5f5dc")
    ax.coastlines(resolution="10m", linewidth=0.5)

    for s in range(pred_coords.shape[0]):
        ax.plot(pred_coords[s, :, 1], pred_coords[s, :, 0],
                color="grey", alpha=0.35, linewidth=0.8)
    mean_lat = jnp.nanmean(pred_coords[..., 0], axis=0)
    mean_lon = jnp.nanmean(pred_coords[..., 1], axis=0)
    ax.plot(mean_lon, mean_lat, color="C3", linewidth=2, label="pred mean")

    if true_coords is not None:
        ax.plot(true_coords[:, 1], true_coords[:, 0],
                color="k", linewidth=2, label="ground truth")
        ax.plot(true_coords[0, 1], true_coords[0, 0], "ko", markersize=6)
    if start_position is not None:
        ax.plot(start_position[1], start_position[0], "ko", markersize=6,
                label="start")

    all_lat = pred_coords[..., 0].ravel()
    all_lon = pred_coords[..., 1].ravel()
    if true_coords is not None:
        all_lat = jnp.concatenate([all_lat, true_coords[:, 0].ravel()])
        all_lon = jnp.concatenate([all_lon, true_coords[:, 1].ravel()])
    pad = 0.3
    ax.set_extent([float(jnp.nanmin(all_lon)) - pad, float(jnp.nanmax(all_lon)) + pad,
                   float(jnp.nanmin(all_lat)) - pad, float(jnp.nanmax(all_lat)) + pad])
    ax.legend(loc="best", frameon=False)
    ax.set_title(title)
    plt.show()

plot_prediction(pred_coords, true_coords=true_coords,
                title=f"CoDiT-GRU 72 h prediction — {DEMO_NAME}")

## 2. Predict on your own region

To predict somewhere else, edit the constants below and run the cells. The
region should be over open ocean (the shipped model was trained on
subtropical gyres) and large enough that the Global Drifter Program has at
least one drifter in it during the chosen window — a 2–3° box over one to
two weeks is a good starting point.

Because the download hits four separate services, expect a few minutes on
the first run per new region. Everything is cached under
`$CODIT_DATA_PATH/datasets/example_datasets/<your_name>/`.


In [ ]:
def download_region(name, center_lat, center_lon, coord_delta, start, end):
    """Download context + drifters for one region and return an OceanDataset."""
    min_lat, max_lat = center_lat - coord_delta, center_lat + coord_delta
    min_lon, max_lon = center_lon - coord_delta, center_lon + coord_delta

    cfg_dir = os.path.join(config_paths.CONFIG_PATH, "example_datasets")
    os.makedirs(cfg_dir, exist_ok=True)

    raw = {
        "name": name,
        "latitudes": [min_lat, max_lat],
        "longitudes": [min_lon, max_lon],
        "start_date": start,
        "end_date": end,
        "context_delta": 0.5,
        "description": f"user region: {name}",
    }
    config_funcs.save_config(raw, cfg_dir, config_paths.DATA_PATH)
    cfg = config_funcs.load_config(name, cfg_dir)

    print(f"Downloading data for {name} …")
    dataset_download.download_drifter(cfg)
    dataset_download.download_hycom(cfg)
    dataset_download.download_era5(cfg)
    dataset_download.download_copernicus(cfg)

    return OceanDataset(cfg, length=PREDICTION_LENGTH,
                        num_grid_points=16, delta_metres=10000,
                        interp_method="linear")


def predict_from_first_drifter(ds, name, num_samples=20):
    """Predict from the first valid drifter start in ds and plot vs. ground truth."""
    if len(ds) == 0:
        raise RuntimeError(
            f"No drifter starts in {name}. Widen the date range or bounding box, "
            "or use `predict_point()` in Section 3 to forecast from a coordinate you pick."
        )
    start_position, start_time, _times, interp, true_coords = ds[0]
    print(f"Predicting from ({start_position[0]:.3f}, {start_position[1]:.3f}) "
          f"at {start_time} …")
    pred_coords = model.predict(
        start_position, start_time, PREDICTION_LENGTH, interp, ds,
        num_samples=num_samples, key=key,
    )
    plot_prediction(pred_coords, true_coords=true_coords,
                    title=f"CoDiT-GRU 72 h prediction — {name}")
    return true_coords, pred_coords

In [ ]:
# Example: North Pacific subtropical gyre, one week in spring 2021.
# Change these four constants and re-run to try somewhere else.
my_ds = download_region(
    name="predict_myregion_npacific",
    center_lat=28.0,
    center_lon=-140.0,
    coord_delta=1.5,
    start=datetime(2021, 4, 15),
    end=datetime(2021, 4, 22),
)
true_coords, pred_coords = predict_from_first_drifter(my_ds, "predict_myregion_npacific")

## 3. Point-and-shoot: predict from a chosen coordinate and time

The model doesn't need a drifter to exist at your chosen location — it only
needs the context variables (currents + wind) at the right space-time patch.
Once a region has been downloaded (as in Sections 1 or 2), the same
`OceanDataset` can supply the interpolator for any starting position inside
its bounding box, at any datetime inside its date range.

Give it a `(lat, lon, datetime)` and it returns 72 h of predicted samples.

**Constraints on the pick:**
- The starting position must lie inside the region's bounding box, with a
  bit of margin so 72 h of drift stays inside.
- `start_time + 72 h` must fit inside the downloaded date range (leave one
  day of margin at each end).


In [ ]:
def predict_point(ds, start_lat, start_lon, start_time, num_samples=20,
                  title=None):
    """Forecast 72 h from a chosen (lat, lon, start_time).

    ``ds`` must be an :class:`OceanDataset` whose bounding box contains
    ``(start_lat, start_lon)`` and whose date range contains
    ``[start_time, start_time + 72 h]``.
    """
    start_position = jnp.asarray([start_lat, start_lon])
    interp = ds.get_interpolator(
        min_date=start_time - timedelta(hours=1),
        max_date=start_time + timedelta(hours=PREDICTION_LENGTH + 1),
    )
    pred_coords = model.predict(
        start_position, start_time, PREDICTION_LENGTH, interp, ds,
        num_samples=num_samples, key=key,
    )
    plot_prediction(
        pred_coords, start_position=start_position,
        title=title or f"CoDiT-GRU 72 h from ({start_lat:.2f}, {start_lon:.2f}) at {start_time}",
    )
    return pred_coords

In [ ]:
# Reuse the N. Pacific region from Section 2 and predict from a hand-picked point.
pred_from_point = predict_point(
    my_ds,
    start_lat=28.5,
    start_lon=-139.5,
    start_time=datetime(2021, 4, 17, 12, 0),
    num_samples=20,
)

## What's next?

- **Train from scratch on your own region** — see
  [`02_train_quickstart.ipynb`](02_train_quickstart.ipynb) for a small
  training loop and [`03_prepare_your_own_region.ipynb`](03_prepare_your_own_region.ipynb)
  for the full download → preprocess → train → evaluate pipeline.
- **Batch evaluation on Zenodo splits** — after
  `python scripts/user/setup/download_data.py --dataset open_ocean`,
  use `python scripts/user/eval/codit_eval.py --dataset open_ocean --split test`.
- **Data format** — see [`docs/data_format.md`](../../docs/data_format.md)
  for the 8-channel context tensor layout and how to extend it.
